# trajectory-kit: Developer Guide

This notebook covers the internals that power trajectory-kit: the query vocabulary, the stochastic planner, the iterator layer, trajectory metadata, and the domain registry contract. It assumes you have already read the basic tutorial.

The same 20-atom synthetic system is used throughout (ALA-GLY-PRO backbone + 2× TIP3 water, 5-frame DCD).

---


## 0. Setup

In [1]:
import struct
import tempfile
import pprint
from pathlib import Path
import numpy as np

tmp = Path(tempfile.mkdtemp())

pdb_text = """\
ATOM      1  N   ALA A   1       1.000   5.000   5.000  1.00  0.00      PROT
ATOM      2  HN  ALA A   1       0.100   5.800   5.300  1.00  0.00      PROT
ATOM      3  CA  ALA A   1       2.400   5.000   5.000  1.00  0.00      PROT
ATOM      4  CB  ALA A   1       3.000   6.300   5.200  1.00  0.00      PROT
ATOM      5  C   ALA A   1       3.200   3.900   4.600  1.00  0.00      PROT
ATOM      6  O   ALA A   1       2.800   2.800   4.400  1.00  0.00      PROT
ATOM      7  N   GLY A   2       4.600   4.100   4.500  1.00  0.00      PROT
ATOM      8  HN  GLY A   2       5.000   5.000   4.700  1.00  0.00      PROT
ATOM      9  CA  GLY A   2       5.800   3.200   4.100  1.00  0.00      PROT
ATOM     10  C   GLY A   2       7.200   3.500   4.000  1.00  0.00      PROT
ATOM     11  O   GLY A   2       7.700   4.600   4.200  1.00  0.00      PROT
ATOM     12  N   PRO A   3       8.100   2.500   3.700  1.00  0.00      PROT
ATOM     13  CA  PRO A   3       9.500   2.800   3.600  1.00  0.00      PROT
ATOM     14  C   PRO A   3      10.300   1.900   3.200  1.00  0.00      PROT
ATOM     15  OH2 TIP B   4      12.000   5.000   5.000  1.00  0.00      SOLV
ATOM     16  H1  TIP B   4      12.900   5.500   5.100  1.00  0.00      SOLV
ATOM     17  H2  TIP B   4      11.300   5.700   4.800  1.00  0.00      SOLV
ATOM     18  OH2 TIP B   5      14.000   3.000   5.500  1.00  0.00      SOLV
ATOM     19  H1  TIP B   5      14.800   3.600   5.400  1.00  0.00      SOLV
ATOM     20  H2  TIP B   5      13.400   3.700   5.200  1.00  0.00      SOLV
END
"""

psf_text = """\
PSF

       1 !NTITLE
 REMARKS synthetic 20-atom ALA-GLY-PRO + 2xTIP3

      20 !NATOM
       1 PROT     1 ALA  N    NH1     -0.47000     14.007           0
       2 PROT     1 ALA  HN   H        0.31000      1.008           0
       3 PROT     1 ALA  CA   CT1     -0.02000     12.011           0
       4 PROT     1 ALA  CB   CT3     -0.27000     12.011           0
       5 PROT     1 ALA  C    C        0.51000     12.011           0
       6 PROT     1 ALA  O    O       -0.51000     15.999           0
       7 PROT     2 GLY  N    NH1     -0.47000     14.007           0
       8 PROT     2 GLY  HN   H        0.31000      1.008           0
       9 PROT     2 GLY  CA   CT2     -0.18000     12.011           0
      10 PROT     2 GLY  C    C        0.51000     12.011           0
      11 PROT     2 GLY  O    O       -0.51000     15.999           0
      12 PROT     3 PRO  N    N       -0.29000     14.007           0
      13 PROT     3 PRO  CA   CP1     -0.02000     12.011           0
      14 PROT     3 PRO  C    C        0.51000     12.011           0
      15 SOLV     4 TIP  OH2  OT      -0.83400     15.999           0
      16 SOLV     4 TIP  H1   HT       0.41700      1.008           0
      17 SOLV     4 TIP  H2   HT       0.41700      1.008           0
      18 SOLV     5 TIP  OH2  OT      -0.83400     15.999           0
      19 SOLV     5 TIP  H1   HT       0.41700      1.008           0
      20 SOLV     5 TIP  H2   HT       0.41700      1.008           0

      17 !NBOND: bonds
       1       2       1       3       3       4       3       5
       5       6       5       7       7       8       7       9
       9      10      10      11      10      12      12      13
      13      14      15      16      15      17      18      19
      18      20

       0 !NTHETA: angles

       0 !NPHI: dihedrals

       0 !NIMPHI: impropers

"""

def write_dcd(path, n_atoms=20, n_frames=5):
    def record(payload):
        n = struct.pack('<i', len(payload))
        return n + payload + n
    icntrl = [0] * 20
    icntrl[0] = n_frames
    hdr = b'CORD' + struct.pack('<20i', *icntrl)
    hdr += b'\x00' * (84 - len(hdr))
    title = struct.pack('<i', 1) + b'synthetic 20-atom trajectory    '
    natom = struct.pack('<i', n_atoms)
    x0 = np.array([ 1.0, 0.1, 2.4, 3.0, 3.2, 2.8,
                    4.6, 5.0, 5.8, 7.2, 7.7, 8.1,
                    9.5,10.3,12.0,12.9,11.3,14.0,14.8,13.4], dtype=np.float32)
    y0 = np.array([ 5.0, 5.8, 5.0, 6.3, 3.9, 2.8,
                    4.1, 5.0, 3.2, 3.5, 4.6, 2.5,
                    2.8, 1.9, 5.0, 5.5, 5.7, 3.0, 3.6, 3.7], dtype=np.float32)
    z0 = np.array([ 5.0, 5.3, 5.0, 5.2, 4.6, 4.4,
                    4.5, 4.7, 4.1, 4.0, 4.2, 3.7,
                    3.6, 3.2, 5.0, 5.1, 4.8, 5.5, 5.4, 5.2], dtype=np.float32)
    with open(path, 'wb') as f:
        f.write(record(hdr))
        f.write(record(title))
        f.write(record(natom))
        for fi in range(n_frames):
            xs = x0 + float(fi)
            f.write(record(xs.tobytes()))
            f.write(record(y0.tobytes()))
            f.write(record(z0.tobytes()))

pdb_path = tmp / 'synthetic.pdb'
psf_path = tmp / 'synthetic.psf'
dcd_path = tmp / 'synthetic.dcd'

pdb_path.write_text(pdb_text, encoding='ascii')
psf_path.write_text(psf_text, encoding='ascii')
write_dcd(dcd_path)

from trajectory_kit import sim as Sim
s = Sim(typing=pdb_path, topology=psf_path, trajectory=dcd_path)

print('Ready.')


Ready.


---
## 1. The Full Query Vocabulary

Below are the complete query templates for each domain with every key shown explicitly.
In practice you only include the keys you want to filter on — the rest are simply omitted.
Empty include means accept all; empty exclude means reject nothing.


In [2]:
# ── TYPING FILE (PDB) ─────────────────────────────────────────────────────
TYPE_QUERY_FULL = {

    # Categorical: bare set = include only; (inc_set, exc_set) = two-sided
    "atom_name":    (set(), set()),
    "residue_name": (set(), set()),
    "segment_name": (set(), set()),

    # Numeric ranges: bare (lo, hi) = include range; ((lo,hi),(lo,hi)) = two-sided
    # Either bound may be None for open-ended.
    "local_ids":          (None, None),   # atom serial number
    "residue_ids":        (None, None),   # residue sequence number
    "occupancy":          (None, None),
    "temperature_coeff":  (None, None),

    # Coordinate ranges (Å)
    "x": (None, None),
    "y": (None, None),
    "z": (None, None),
}


In [3]:
# ── TOPOLOGY FILE (PSF) ───────────────────────────────────────────────────
TOPO_QUERY_FULL = {

    # Categorical
    "atom_name":    (set(), set()),
    "residue_name": (set(), set()),
    "segment_name": (set(), set()),
    "atom_type":    (set(), set()),

    # Numeric ranges
    "local_ids":   (None, None),
    "residue_ids": (None, None),
    "charge":      (None, None),
    "mass":        (None, None),
    "is_virtual":  (None, None),

    # Drude polarisable force field only (ignored on standard PSF)
    "drude_alpha": (None, None),
    "drude_thole": (None, None),

    # Bond graph constraints
    "bonded_with": (
        [   # include blocks — each block is a constraint the atom must satisfy
            # {"neighbor": {<sub-query>}, "count": {"eq"|"ge"|"le"|"gt"|"lt": n}}
            # {"total": True,             "count": {"eq": 4}}  <- total degree
        ],
        [   # exclude blocks — same structure, matching atoms are removed
        ],
    ),
    # "all": atom must satisfy every include block
    # "any": atom must satisfy at least one include block
    "bonded_with_mode": ("all", None),
}


In [4]:
# ── TRAJECTORY FILE (DCD) ─────────────────────────────────────────────────
TRAJ_QUERY_FULL = {

    # Pre-resolved global atom indices (injected by the framework for positions())
    # Format: (list[int], set())  — the set is reserved for future exclude support
    "global_ids": ([], set()),

    # (start, stop, step) — both start and stop are INCLUSIVE
    # Omit entirely to read all frames.
    "frame_interval": (0, None, 1),
}


---
## 2. The Iterator Layer — `iter_records` and `iter_records_sample`

All text-based parsers (PDB, PSF, XYZ) are built on two generic iterators in `file_parse_help.py`.
Understanding them is essential if you want to add a new parser.

### `iter_records` — exact, streaming

Two modes:
- **`counted`** — finds a header line, reads exactly N following lines. Used by PSF (`!NATOM`) and XYZ (atom count on line 1).
- **`predicate`** — yields every line that satisfies a predicate. Used by PDB (`ATOM`/`HETATM` lines).

The key property is that it is a **generator** — it never loads the full file into memory.


In [5]:
from trajectory_kit import file_parse_help as fph

# Predicate mode: stream every ATOM/HETATM line from the PDB
atom_iter = fph.iter_records(
    pdb_path,
    mode='predicate',
    record_pred=lambda line: line[0:6] in ('ATOM  ', 'HETATM'),
    parse_row=lambda line, i: {
        'global_id':   i,
        'atom_name':   line[12:16].strip(),
        'residue_name': line[17:21].strip(),
        'segment_name': line[72:76].strip(),
    },
    start_index=0,
    encoding='ascii',
    errors='replace',
)

records = list(atom_iter)
print(f'Records read: {len(records)}')   # 20
print('First record:', records[0])
print('Last record: ', records[-1])


Records read: 20
First record: {'global_id': 0, 'atom_name': 'N', 'residue_name': 'ALA', 'segment_name': 'PROT'}
Last record:  {'global_id': 19, 'atom_name': 'H2', 'residue_name': 'TIP', 'segment_name': 'SOLV'}


In [6]:
# Counted mode: PSF - find !NATOM header, read exactly that many lines
psf_iter = fph.iter_records(
    psf_path,
    mode='counted',
    header_pred=lambda line: '!NATOM' in line,
    count_from_header=lambda line: int(line.split()[0]),
    parse_row=lambda line, i: {
        'global_id':   i,
        'atom_name':   line.split()[4],
        'atom_type':   line.split()[5],
        'charge':      float(line.split()[6]),
    },
    start_index=0,
)

psf_records = list(psf_iter)
print(f'PSF atoms read: {len(psf_records)}')   # 20
print('Types seen:', sorted({r["atom_type"] for r in psf_records}))


PSF atoms read: 20
Types seen: ['C', 'CP1', 'CT1', 'CT2', 'CT3', 'H', 'HT', 'N', 'NH1', 'O', 'OT']


### `iter_records_sample` — stochastic single-pass sampler

This is what powers the planner. For large files it is impractical to read everything just to estimate how many atoms a query will return. `iter_records_sample` does a **single pass** with geometric-skip Bernoulli sampling:

- Counts total lines in one pass
- Computes `p = min(1.0, target_sample_size / n_lines)`
- Uses geometric skip to jump between sampled lines in O(1) per sample — no per-line random draw
- Returns rich metadata alongside the sampled records

On a 20-atom file `p = 1.0` (full sample). On a real file with 100 000 atoms the same call samples ~3 000 records regardless of file size.


In [7]:
sample_info = fph.iter_records_sample(
    pdb_path,
    record_pred=lambda line: line[0:6] in ('ATOM  ', 'HETATM'),
    parse_row=lambda line, i: {'global_id': i, 'atom_name': line[12:16].strip()},
    target_sample_size=3000,
    rng_seed=42,
)

print('Total lines in file:       ', sample_info['number_of_lines'])
print('Lines physically sampled:  ', sample_info['number_of_sampled_lines'])
print('Eligible records sampled:  ', sample_info['number_of_sampled_eligible_records'])
print()
print('Sampling metadata:')
pprint.pprint(sample_info['sampling_metadata'])
print()
print('First 3 sampled records:')
pprint.pprint(sample_info['sampled_records'][:3])


Total lines in file:        21
Lines physically sampled:   21
Eligible records sampled:   20

Sampling metadata:
{'rng_seed': 42,
 'sample_probability': 1.0,
 'sampling_mode': 'bernoulli_skip',
 'target_sample_size': 3000}

First 3 sampled records:
[{'atom_name': 'N', 'global_id': 0},
 {'atom_name': 'HN', 'global_id': 1},
 {'atom_name': 'CA', 'global_id': 2}]


---
## 3. The Planner — cost estimation before I/O

Every domain has a `_plan_*` function that estimates what a query will return **without executing it**. This is useful for:
- Checking that a query makes sense before committing to a large DCD read
- Estimating memory requirements
- Debugging unexpected selections

### PSF stochastic planner


In [8]:
from trajectory_kit import psf_parse

keys, reqs = psf_parse._get_topology_keys_reqs_psf(psf_path)

# Plan a simple atom-name query
plan = psf_parse._plan_topology_query_psf(
    psf_filepath=psf_path,
    query_dictionary={'atom_name': ({'CA'}, set())},
    request_string='global_ids',
    keywords_available=keys,
    requests_available=reqs,
)

pprint.pprint(plan)


{'confidence': 'low',
 'estimated_bytes': 24,
 'estimated_mib': 2.288818359375e-05,
 'n_atoms': 3,
 'n_lines_eligible': 24,
 'n_lines_matching': 3,
 'n_lines_sampled': 40,
 'planner_mode': 'stochastic'}


Key fields returned by the stochastic planner:

| Field | Meaning |
|-------|---------|
| `planner_mode` | `"stochastic"` for PSF/PDB, `"header"` for DCD |
| `n_lines_sampled` | Number of file lines physically read in the sample pass |
| `n_lines_eligible` | Lines that passed the record predicate (atom rows only) |
| `n_lines_matching` | Lines that also matched the query predicate |
| `n_atoms` | Estimated atoms the query will return |
| `estimated_bytes` / `estimated_mib` | Predicted output size |
| `confidence` | `none / low / medium / high` — based on how many matches were seen |

The `supported: False` case is important — `bonded_with` constraints require the full bond graph and cannot be estimated stochastically. The planner reports this explicitly rather than silently returning a bad estimate.


In [9]:
# Plan that returns supported=False — bonded_with not estimable stochastically
plan_unsupported = psf_parse._plan_topology_query_psf(
    psf_filepath=psf_path,
    query_dictionary={
        'bonded_with': (
            [{'total': True, 'count': {'eq': 3}}],
            [],
        ),
        'bonded_with_mode': ('all', None),
    },
    request_string='global_ids',
    keywords_available=keys,
    requests_available=reqs,
)

print('supported:', plan_unsupported['supported'])
print('reason:   ', plan_unsupported['reason'])


supported: False
reason:    PSF stochastic planning does not estimate bonded_with constraints. Only direct atom-selection prevalence is estimated.


In [10]:
# PDB planner — same interface
from trajectory_kit import pdb_parse

pdb_keys, pdb_reqs = pdb_parse._get_type_keys_reqs_pdb(pdb_path)

# Query: all SOLV atoms
plan_solv = pdb_parse._plan_type_query_pdb(
    pdb_filepath=pdb_path,
    query_dictionary={'segment_name': ({'SOLV'}, set())},
    request_string='global_ids',
    keywords_available=pdb_keys,
    requests_available=pdb_reqs,
)

pprint.pprint(plan_solv)
# n_atoms should estimate ~6 (the 6 SOLV atoms out of 20)

# High-selectivity query: single atom type (OT — water oxygens, 2 out of 20)
plan_ot = pdb_parse._plan_type_query_pdb(
    pdb_filepath=pdb_path,
    query_dictionary={'atom_name': ({'OH2'}, set())},
    request_string='global_ids',
    keywords_available=pdb_keys,
    requests_available=pdb_reqs,
)
print('\nOH2 query:')
print('  n_atoms estimated:', plan_ot['n_atoms'])
print('  confidence:       ', plan_ot['confidence'])
# confidence will be low/none on this small file; high on real 100k-atom systems


{'confidence': 'low',
 'estimated_bytes': 48,
 'estimated_mib': 4.57763671875e-05,
 'n_atoms': 6,
 'n_lines_eligible': 20,
 'n_lines_matching': 6,
 'n_lines_sampled': 21,
 'planner_mode': 'stochastic'}

OH2 query:
  n_atoms estimated: 2
  confidence:        low


### DCD header planner

DCD uses a different strategy — instead of sampling, it reads the binary header (a handful of bytes) and computes exact frame counts from the trajectory metadata. No atom data is read at all.


In [11]:
from trajectory_kit import dcd_parse

dcd_keys, dcd_reqs = dcd_parse._get_trajectory_keys_reqs_dcd(dcd_path)

dcd_plan = dcd_parse._plan_trajectory_query_dcd(
    dcd_filepath=dcd_path,
    query_dictionary={
        'global_ids': ([2, 8, 12], set()),    # 3 CA atoms
        'frame_interval': (1, 3, 1),          # frames 1, 2, 3
    },
    request_string='positions',
    keywords_available=dcd_keys,
    requests_available=dcd_reqs,
)

pprint.pprint(dcd_plan)
# n_frames = 3, n_atoms = 3, estimated_bytes = 3*3*3*4 = 108


{'estimated_bytes': 108,
 'estimated_mib': 0.000102996826171875,
 'n_atoms': 3,
 'n_frames': 3,
 'planner_mode': 'header'}


### Using `planFlag` from the `sim` interface

`positions()` exposes the planner directly via `planFlag=True`. The TYPE/TOPO selection still runs (to resolve `global_ids`) but the DCD is never opened.


In [12]:
plan_out = s.positions(
    TYPE_Q={'atom_name': 'CA'},
    TRAJ_Q={'frame_interval': (0, 4, 1)},
    planFlag=True,
)

print('payload is None (DCD not read):', plan_out['payload'] is None)
print()
print('selection:')
pprint.pprint(plan_out['selection'])
print()
print('trajectory plan:')
pprint.pprint(plan_out['plan'])


payload is None (DCD not read): True

selection:
{'count': None, 'merge_mode': 'intersection', 'sources': []}

trajectory plan:
{'estimated_bytes': 0,
 'estimated_mib': 0.0,
 'n_atoms': 0,
 'n_frames': 5,
 'planner_mode': 'header'}


---
## 4. Trajectory Metadata

`get_trajectory()` returns a tuple `(positions, meta)`. The `meta` dict tells you exactly what was read — useful for verifying frame interval arithmetic and debugging off-by-one errors.


In [13]:
pos, meta = s.get_trajectory(
    QUERY={
        'global_ids': (list(range(20)), set()),  # all atoms
        'frame_interval': (1, 4, 2),             # frames 1 and 3
    },
    REQUEST='positions',
)

print('positions shape:', pos.shape)   # (2, 20, 3)
print('dtype:          ', pos.dtype)   # float32
print()
print('metadata:')
pprint.pprint(meta)


positions shape: (2, 20, 3)
dtype:           float32

metadata:
{'first_frame_read': 1,
 'last_frame_read': 3,
 'n_frames_read': 2,
 'start': 1,
 'start_inclusive': True,
 'step': 2,
 'stop': 5,
 'stop_inclusive': False}


| Meta field | Meaning |
|------------|---------|
| `first_frame_read` | Index of the first frame actually read |
| `last_frame_read` | Index of the last frame actually read |
| `n_frames_read` | Total frames in the output array |
| `start / stop / step` | Resolved interval in Python range semantics (stop exclusive) |
| `start_inclusive` | Always `True` — the user-facing start is inclusive |
| `stop_inclusive` | Always `False` — internally converted to exclusive stop |

The stop value in `meta` is always the user-facing stop + 1 (Python range convention). On this file `frame_interval=(1, 4, 2)` → `meta['stop'] = 5`, `meta['step'] = 2`, giving frames [1, 3].


---
## 5. Reading DCD Header Metadata Directly

If you just need file-level metadata without constructing a full `sim`, the DCD header reader is available directly.


In [14]:
nset, natom, has_unitcell = dcd_parse._read_dcd_header_metadata(dcd_path)

print('Total frames:  ', nset)          # 5
print('Total atoms:   ', natom)         # 20
print('Has unit cell: ', has_unitcell)  # False


Total frames:   5
Total atoms:    20
Has unit cell:  False


---
## 6. Confidence Tiers in the Stochastic Planner

The PSF and PDB planners report a `confidence` field based on how many matching records were seen in the sample:

| Confidence | Matching records in sample | Interpretation |
|------------|---------------------------|----------------|
| `none` | 0 | Query likely returns nothing, or no eligible records in file |
| `low` | 1–9 | Rough order-of-magnitude guess |
| `medium` | 10–99 | Reasonable for capacity planning |
| `high` | 100+ | Reliable estimate |

On a 20-atom synthetic file most queries will return `low` or `none` confidence because `n_matching_sampled` is inherently small. On real files with tens of thousands of atoms and `target_sample_size=3000`, most queries hit `high` confidence. The confidence tier degrades gracefully for very selective queries (e.g. a single rare atom type).


In [15]:
# Confidence on a selective query (OT — 2 atoms out of 20)
plan_ot = psf_parse._plan_topology_query_psf(
    psf_filepath=psf_path,
    query_dictionary={'atom_type': ({'OT'}, set())},
    request_string='global_ids',
    keywords_available=keys,
    requests_available=reqs,
)

print('OT query:')
print('  n_lines_matching:', plan_ot['n_lines_matching'])
print('  n_atoms:         ', plan_ot['n_atoms'])
print('  confidence:      ', plan_ot['confidence'])

# Confidence on a broad query (all PROT atoms — 14 out of 20)
plan_prot = psf_parse._plan_topology_query_psf(
    psf_filepath=psf_path,
    query_dictionary={'segment_name': ({'PROT'}, set())},
    request_string='global_ids',
    keywords_available=keys,
    requests_available=reqs,
)

print()
print('PROT segment query:')
print('  n_lines_matching:', plan_prot['n_lines_matching'])
print('  n_atoms:         ', plan_prot['n_atoms'])
print('  confidence:      ', plan_prot['confidence'])


OT query:
  n_lines_matching: 2
  n_atoms:          2
  confidence:       low

PROT segment query:
  n_lines_matching: 14
  n_atoms:          14
  confidence:       medium


---
## 7. Adding a New Parser — the Contract

The domain registry in `main.py` resolves function names by template at runtime:

```python
"keys_fn_template":   "_get_{domain}_keys_reqs_{fmt}"
"plan_fn_template":   "_plan_{domain}_query_{fmt}"
"query_fn_template":  "_get_{domain}_query_{fmt}"
"update_fn_template": "_update_{domain}_globals_{fmt}"
```

To add XTC trajectory support you would write `xtc_parse.py` with exactly these four functions, then add `".xtc"` to the trajectory `supported_formats` set. `_validate_domain_module_contract` checks the contract at import time and raises `AttributeError` with a clear message if any function is missing — you will never silently load a broken parser.


In [16]:
# Inspect the domain registry directly
import pprint
pprint.pprint(s._domain_registry)


{'topology': {'file_attr': 'top_file',
              'keys_attr': 'topo_file_keys',
              'keys_fn_template': '_get_topology_keys_reqs_{fmt}',
              'label': 'topology',
              'plan_fn_template': '_plan_topology_query_{fmt}',
              'query_fn_template': '_get_topology_query_{fmt}',
              'reqs_attr': 'topo_file_reqs',
              'supported_formats': {'.psf', '.mae'},
              'type_attr': 'top_type',
              'update_fn_template': '_update_topology_globals_{fmt}'},
 'trajectory': {'file_attr': 'traj_file',
                'keys_attr': 'traj_file_keys',
                'keys_fn_template': '_get_trajectory_keys_reqs_{fmt}',
                'label': 'trajectory',
                'plan_fn_template': '_plan_trajectory_query_{fmt}',
                'query_fn_template': '_get_trajectory_query_{fmt}',
                'reqs_attr': 'traj_file_reqs',
                'supported_formats': {'.dcd'},
                'type_attr': 'traj_type',
       

---
## 8. `fetch()` — the general-purpose extraction method

`positions()` is a convenience wrapper. For any non-coordinate payload — atom names, charges, masses, residue IDs — use `fetch()`.
It also handles cross-domain intersections: you can combine a TYPE_Q atom filter with a TOPO_R charge request, and the framework resolves the intersection automatically.


In [17]:
# Fetch charges for PROT CA atoms:
# TYPE_Q selects CA atoms → [2, 8, 12]
# TOPO_R returns their charges from PSF
result = s.fetch(
    TYPE_Q={'atom_name': 'CA'},
    TOPO_R='charges',
)

print('Mode:          ', result['mode'])
print('Atoms selected:', result['selection']['count'])    # 3
print('Sources:       ', result['selection']['sources'])  # ['typing']
print('CA charges:    ', result['payload']['topology'])

# Fetch atom types for negatively charged PROT atoms
result2 = s.fetch(
    TOPO_Q={
        'segment_name': 'PROT',
        'charge': (None, -0.1),
    },
    TOPO_R='atom_types',
)
print()
print('Neg-charged PROT atom types:', result2['payload']['topology'])


Mode:           fetch
Atoms selected: 3
Sources:        ['typing']
CA charges:     [-0.02, -0.18, -0.02]

Neg-charged PROT atom types: ['NH1', 'CT3', 'O', 'NH1', 'CT2', 'O', 'N']
